# ARC-AGI — Category Explorer (Colab)

Browse all 400 training tasks by human-decision-tree category.  
Where a rule-based solver exists, predictions are shown alongside the expected output.

**Run order:** Cell 1 (setup) → Cell 2 (classify) → then any of Cells 3–5 in any order.

In [ ]:
# ── Cell 1: Setup — works in Colab and locally ───────────────────────────────
import os, io, subprocess, urllib.request, zipfile, sys
from pathlib import Path

IN_COLAB    = 'google.colab' in sys.modules
GITHUB_USER = 'rodehyde'
REPO        = 'arc-agi-solver'

if IN_COLAB:
    REPO_DIR = f'/content/{REPO}'
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', f'https://github.com/{GITHUB_USER}/{REPO}.git', REPO_DIR],
                       check=True)
    else:
        result = subprocess.run(['git', '-C', REPO_DIR, 'pull', '--ff-only'],
                                capture_output=True, text=True)
        print(result.stdout or result.stderr)
    os.chdir(REPO_DIR)
else:
    # Local: notebook lives in notebooks/, repo root is one level up
    REPO_DIR = str(Path(os.getcwd()).parent if Path(os.getcwd()).name == 'notebooks'
                   else Path(os.getcwd()))
    os.chdir(REPO_DIR)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print(f'Working directory: {os.getcwd()}')
print(subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)

train_dir = Path('data/training')
if not train_dir.exists() or len(list(train_dir.glob('*.json'))) < 400:
    print('Downloading ARC-AGI training data...')
    with urllib.request.urlopen(
        'https://github.com/fchollet/ARC-AGI/archive/refs/heads/master.zip') as r:
        raw = r.read()
    train_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        for m in zf.namelist():
            if 'data/training/' in m and m.endswith('.json'):
                (train_dir / Path(m).name).write_bytes(zf.read(m))
print(f'Training tasks available: {len(list(train_dir.glob("*.json")))}')

In [ ]:
# ── Cell 2: Classify all tasks and show category summary ─────────────────────
# NOTE: re-run this cell after every git pull — it reloads all modules.
import importlib, sys, numpy as np, matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from collections import defaultdict
from pathlib import Path

# Force-reload scripts so that git-pulled changes take effect immediately
for mod_name in list(sys.modules):
    if mod_name.startswith('scripts.'):
        importlib.reload(sys.modules[mod_name])

from scripts.human_tree import load_task as ht_load, classify
from scripts.solvers    import (load_task, task_delta, find_solver,
                                ALL_PRIMITIVES, verify)

# ── Quick self-check ──────────────────────────────────────────────────────────
_sample = load_task('00d62c1b')
_te     = _sample['test'][0]
assert _te['output'] is not None, \
    "Test output is None — training data may be missing outputs. Re-run Cell 1."
print(f"Self-check OK: test output present (shape {_te['output'].shape})")

# ── ARC colour palette ────────────────────────────────────────────────────────
ARC_COLORS = ['#000000','#0074D9','#FF4136','#2ECC40','#FFDC00',
              '#AAAAAA','#F012BE','#FF851B','#7FDBFF','#870C25']
_cmap = ListedColormap(ARC_COLORS)
_norm = BoundaryNorm(boundaries=list(range(11)), ncolors=10)

# ── Display helpers ───────────────────────────────────────────────────────────
def _shape_str(grid):
    """Return 'RxC' string for a grid."""
    return f'{grid.shape[0]}×{grid.shape[1]}'

def _show_grid(ax, grid, title='', border_colour=None):
    ax.imshow(np.array(grid, dtype=np.uint8), cmap=_cmap, norm=_norm,
              interpolation='nearest')
    ax.set_title(title, fontsize=8, pad=3)
    ax.set_xticks([]); ax.set_yticks([])
    if border_colour:
        for sp in ax.spines.values():
            sp.set_visible(True)
            sp.set_edgecolor(border_colour)
            sp.set_linewidth(4)

def _blank_ax(ax, msg=''):
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_facecolor('#f0f0f0')
    for sp in ax.spines.values():
        sp.set_visible(False)
    if msg:
        ax.set_title(msg, fontsize=8)

def show_task(task_id, show_prediction=True):
    """Display all training pairs then all test pairs for a task.

    Layout (one row per pair):
      col 0 : input
      col 1 : expected output  (orange border on test rows)
      col 2 : solver prediction — green ✓ / red ✗ (only when solver matches)
    """
    task  = load_task(task_id)
    train = task['train']
    tests = task['test']
    n_tr  = len(train)
    n_te  = len(tests)
    d     = task_delta(task)

    # Try to find a rule-based solver
    solver_name, _ = find_solver(task) if show_prediction else (None, None)
    solve_fn = None
    if solver_name:
        solve_fn = next(fn for nm, _, fn in ALL_PRIMITIVES if nm == solver_name)

    # Predictions: treat train inputs as test inputs (for verification)
    tr_preds = [None] * n_tr
    te_preds = [None] * n_te
    if solve_fn:
        try:
            r = solve_fn({**task, 'test': [{'input': p['input']} for p in train]})
            if r: tr_preds = r
        except Exception:
            pass
        try:
            r = solve_fn(task)
            if r: te_preds = r
        except Exception:
            pass

    cols   = 3 if solve_fn else 2
    n_rows = n_tr + n_te
    fig, axes = plt.subplots(n_rows, cols, figsize=(cols * 3.2, n_rows * 3.2),
                             squeeze=False)

    all_tr_correct = solve_fn and all(
        p is not None and np.array_equal(p, pair['output'])
        for p, pair in zip(tr_preds, train))

    # ── Training rows ─────────────────────────────────────────────────────────
    for row, pair in enumerate(train):
        inp, out = pair['input'], pair['output']
        _show_grid(axes[row][0], inp,
                   f'Train {row} — input ({_shape_str(inp)})')
        _show_grid(axes[row][1], out,
                   f'Train {row} — expected ({_shape_str(out)})')
        if solve_fn:
            pred    = tr_preds[row]
            correct = pred is not None and np.array_equal(pred, out)
            pred_grid = pred if pred is not None else np.zeros_like(inp)
            _show_grid(axes[row][2], pred_grid,
                       f'Train {row} — {solver_name} {"✓" if correct else "✗"} ({_shape_str(pred_grid)})',
                       border_colour='#2ECC40' if correct else '#FF4136')

    # ── Test rows ─────────────────────────────────────────────────────────────
    for i, tp in enumerate(tests):
        row     = n_tr + i
        inp     = tp['input']
        out     = tp['output']
        has_out = out is not None

        _show_grid(axes[row][0], inp,
                   f'Test {i} — input ({_shape_str(inp)})')

        if has_out:
            _show_grid(axes[row][1], out,
                       f'Test {i} — expected ({_shape_str(out)})',
                       border_colour='#FF851B')
        else:
            _blank_ax(axes[row][1], 'no ground truth')

        if solve_fn:
            pred = te_preds[i]
            if pred is not None:
                correct_te = has_out and np.array_equal(pred, out)
                border = ('#2ECC40' if correct_te else
                          '#FF851B' if not has_out else '#FF4136')
                label  = (f'Test {i} — {solver_name} '
                          f'{"✓" if correct_te else ("(no gt)" if not has_out else "✗")} ({_shape_str(pred)})')
                _show_grid(axes[row][2], pred, label, border_colour=border)
            else:
                _blank_ax(axes[row][2], f'Test {i} — solver failed')

    # ── Title ─────────────────────────────────────────────────────────────────
    cat    = classify(ht_load(task_id))
    status = (f'  [{solver_name} — {"SOLVED" if all_tr_correct else "partial/failed"}]'
              if solve_fn else '')
    delta  = (f'gained={d["zeros_gained"]}  lost={d["zeros_lost"]}  '
              f'recoloured={d["recoloured"]}  new={d["new_colours"]}')
    fig.suptitle(f'{task_id}  [{cat}]{status}\n{delta}', fontsize=9)
    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.show()

# ── Classify all 400 tasks ────────────────────────────────────────────────────
task_ids = sorted(p.stem for p in Path('data/training').glob('*.json'))
by_cat   = defaultdict(list)
for tid in task_ids:
    cat = classify(ht_load(tid))
    by_cat[cat].append(tid)

print(f'Classified {len(task_ids)} tasks into {len(by_cat)} categories\n')
print(f'{"Category":<35} {"Count":>5}  Sample IDs')
print('-' * 80)
for cat, ids in sorted(by_cat.items(), key=lambda x: -len(x[1])):
    sample = '  ' + '  '.join(ids[:4]) + ('...' if len(ids) > 4 else '')
    print(f'  {cat:<33} {len(ids):>5}{sample}')

In [ ]:
# ── Cell 3: Browse a category one task at a time ─────────────────────────────
# Change CATEGORY and IDX then re-run.
# If the category has a solver, a third column shows the prediction.

CATEGORY = 'FILL_REGIONS'   # ← any category from the table above
IDX      = 0                # ← 0-based index within the category

ids = by_cat[CATEGORY]
print(f'{CATEGORY}: {len(ids)} tasks  |  showing index {IDX} → {ids[IDX]}')
show_task(ids[IDX])

In [ ]:
# ── Cell 4: Show all tasks in a category ─────────────────────────────────────
# Useful for scanning an entire category at once.
# Set SHOW_PREDICTIONS=False to speed things up if no solver exists yet.

CATEGORY         = 'FILL_REGIONS'   # ← category to dump
SHOW_PREDICTIONS = True             # ← False = faster, no prediction column

ids = by_cat[CATEGORY]
print(f'{CATEGORY}: {len(ids)} tasks\n')
for tid in ids:
    show_task(tid, show_prediction=SHOW_PREDICTIONS)

In [ ]:
# ── Cell 5: Inspect a specific task by ID ────────────────────────────────────

TASK_ID = '00d62c1b'   # ← paste any 8-character task ID

show_task(TASK_ID)